1. The dataset contains anonymized sensor features (F01–F47), so **feature engineering** was used to extract more useful information. Row-wise statistics such as mean, standard deviation, min, max, and skewness were created along with some interaction features and grouped sensor sums.

2. To remove less useful features, SHAP feature importance from a baseline XGBoost model was used.

3. Hyperparameters for **XGBoost, CatBoost, and LightGBM** were tuned using **Optuna**.

4. The three models were trained using **5-fold Stratified Cross Validation**, and class imbalance was handled using scale_pos_weight / class weights.

5. Predictions from these models were combined using **stacking**, where a **Logistic Regression model** was used as the meta-model. **Isotonic calibration** was applied to improve probability estimates.

6. Finally, different classification thresholds were tested on OOF predictions and the one giving the best F1-score was selected.

The final pipeline achieved **0.9903** **OOF Accuracy** and **0.9877 F1-score**.

In [1]:
#IMPORTS & ENVIRONMENT SETUP

import pandas as pd
import numpy as np
import random
import warnings

# ML Models
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression

# Evaluation & Validation
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.calibration import CalibratedClassifierCV

# Advanced Optimization & Feature Importance
import optuna
import shap

# Reproducibility & Clean Output
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING) 

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    
seed_everything(42)
print("done with the setUp .")


done with the setUp .


In [2]:
# FEATURE ENGINEERING & SHAP PRUNING

train_path = '/kaggle/input/datasets/mayankbhatt1369/mlarena/ML Challenge Dataset/TRAIN.csv'
test_path  = '/kaggle/input/datasets/mayankbhatt1369/mlarena/ML Challenge Dataset/TEST.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X = train_df.drop('Class', axis=1)
y = train_df['Class']
X_test = test_df.drop('ID', axis=1)
test_ids = test_df['ID']

# 1. Class Imbalance Handling
class_counts = y.value_counts()
scale_pos_weight = class_counts[0] / class_counts[1]
print(f"Target Distribution:\n{class_counts}\nScale Pos Weight: {scale_pos_weight:.4f}\n")

# 2. Advanced Feature Engineering
def engineer_features(df):
    df_eng = df.copy()
    f_cols = [c for c in df.columns if c.startswith('F')]
    
    # Row statics
    df_eng['row_mean'] = df[f_cols].mean(axis=1)
    df_eng['row_std'] = df[f_cols].std(axis=1)
    df_eng['row_max'] = df[f_cols].max(axis=1)
    df_eng['row_min'] = df[f_cols].min(axis=1)
    df_eng['row_skew'] = df[f_cols].skew(axis=1).fillna(0)
    
    # Grouped sensor sums
    df_eng['sum_F01_F10'] = df[[f'F{i:02d}' for i in range(1, 11)]].sum(axis=1)
    df_eng['sum_F11_F20'] = df[[f'F{i:02d}' for i in range(11, 21)]].sum(axis=1)
    df_eng['sum_F21_F30'] = df[[f'F{i:02d}' for i in range(21, 31)]].sum(axis=1)
    
    # Interaction features (First 10 sensors)
    for i in range(1, 10):
        c1, c2 = f'F{i:02d}', f'F{i+1:02d}'
        df_eng[f'diff_{c1}_{c2}'] = df[c1] - df[c2]
        df_eng[f'ratio_mean_{c1}'] = df[c1] / (df_eng['row_mean'] + 1e-6)
        # Polynomial interaction
        df_eng[f'poly_{c1}_{c2}'] = df[c1] * df[c2]
        
    return df_eng

print("Generating features...")
X = engineer_features(X)
X_test = engineer_features(X_test)

# 3. SHAP based feature selection
print(f"Shape before SHAP pruning: {X.shape}")
X_shap, _, y_shap, _ = train_test_split(X, y, test_size=0.5, stratify=y, random_state=42)
shap_model = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=100, random_state=42).fit(X_shap, y_shap)

explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_shap)
if isinstance(shap_values, list): shap_values = shap_values[1] # Handle different SHAP versions

# Keep features that contribute > 0 to the model
mean_abs_shap = np.abs(shap_values).mean(axis=0)
important_features = X.columns[mean_abs_shap > 0.0001].tolist()

X = X[important_features]
X_test = X_test[important_features]
print(f"Shape after SHAP pruning: {X.shape} (Removed {len(mean_abs_shap) - len(important_features)} noisy features)")

Target Distribution:
Class
0    26465
1    17311
Name: count, dtype: int64
Scale Pos Weight: 1.5288

Generating features...
Shape before SHAP pruning: (43776, 82)
Shape after SHAP pruning: (43776, 82) (Removed 0 noisy features)


In [3]:
#OPTUNA HYPERPARAMETER TUNING

print("Running Optuna Tuning ..")
X_t, X_v, y_t, y_v = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# XGBoost Tuning
def objective_xgb(trial):
    params = {
        'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda',
        'scale_pos_weight': scale_pos_weight, 'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
    }
    model = xgb.XGBClassifier(**params, n_estimators=300)
    model.fit(X_t, y_t, eval_set=[(X_v, y_v)], verbose=False)
    preds = (model.predict_proba(X_v)[:, 1] > 0.5).astype(int)
    return f1_score(y_v, preds)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=15)
best_xgb_params = study_xgb.best_params
best_xgb_params.update({'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': scale_pos_weight, 'random_state': 42})

# CatBoost Tuning
def objective_cat(trial):
    params = {
        'task_type': 'GPU', 'eval_metric': 'F1', 'auto_class_weights': 'Balanced', 'verbose': False, 'random_seed': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10)
    }
    model = CatBoostClassifier(**params, iterations=300)
    model.fit(X_t, y_t, eval_set=(X_v, y_v), early_stopping_rounds=50)
    preds = (model.predict_proba(X_v)[:, 1] > 0.5).astype(int)
    return f1_score(y_v, preds)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=15)
best_cat_params = study_cat.best_params
best_cat_params.update({'task_type': 'GPU', 'eval_metric': 'F1', 'auto_class_weights': 'Balanced', 'verbose': False, 'random_seed': 42})

# LightGBM Tuning
def objective_lgb(trial):
    params = {
        'objective': 'binary', 'metric': 'binary_logloss', 'device_type': 'gpu', 'scale_pos_weight': scale_pos_weight, 'verbose': -1, 'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100)
    }
    model = lgb.LGBMClassifier(**params, n_estimators=300)
    model.fit(X_t, y_t, eval_set=[(X_v, y_v)], callbacks=[lgb.early_stopping(50, verbose=False)])
    preds = (model.predict_proba(X_v)[:, 1] > 0.5).astype(int)
    return f1_score(y_v, preds)

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=15)
best_lgb_params = study_lgb.best_params
best_lgb_params.update({'objective': 'binary', 'metric': 'binary_logloss', 'device_type': 'gpu', 'scale_pos_weight': scale_pos_weight, 'verbose': -1, 'random_state': 42})

print("Tuning finisheD,,")

Running Optuna Tuning ..


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Tuning finisheD,,


In [4]:
#K-FOLD TRAINING (LEVEL-1 MODELS)

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_xgb, oof_cat, oof_lgb = np.zeros(len(X)), np.zeros(len(X)), np.zeros(len(X))
test_xgb, test_cat, test_lgb = np.zeros(len(X_test)), np.zeros(len(X_test)), np.zeros(len(X_test))

print(f"Starting {N_SPLITS} fold CV")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # 1. XGBoost
    model_xgb = xgb.XGBClassifier(
        **best_xgb_params,
        n_estimators=1500,
        early_stopping_rounds=100
    )

    model_xgb.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    test_xgb += model_xgb.predict_proba(X_test)[:, 1] / N_SPLITS
    
    # 2. CatBoost
    model_cat = CatBoostClassifier(**best_cat_params, iterations=1500)
    model_cat.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    oof_cat[val_idx] = model_cat.predict_proba(X_val)[:, 1]
    test_cat += model_cat.predict_proba(X_test)[:, 1] / N_SPLITS
    
    # 3. LightGBM
    model_lgb = lgb.LGBMClassifier(**best_lgb_params, n_estimators=1500)
    model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
    test_lgb += model_lgb.predict_proba(X_test)[:, 1] / N_SPLITS
    
    print(f"Fold {fold+1} complete.")

print("Level-1 Out-Of-Fold generation complete.")

Starting 5 fold CV
Fold 1 complete.
Fold 2 complete.
Fold 3 complete.
Fold 4 complete.
Fold 5 complete.
Level-1 Out-Of-Fold generation complete.


In [5]:
# STacking

print("Training stacking model...")
# Create feature matrices from base model predictions
oof_matrix = np.column_stack([oof_xgb, oof_cat, oof_lgb])
test_matrix = np.column_stack([test_xgb, test_cat, test_lgb])

# Level-2 Meta Model
meta_model = LogisticRegression(random_state=42, class_weight='balanced')

# Isotonic Regression
calibrated_meta = CalibratedClassifierCV(meta_model, cv=5, method='isotonic')
calibrated_meta.fit(oof_matrix, y)

# Generate final highly-calibrated probabilities
final_oof_probs = calibrated_meta.predict_proba(oof_matrix)[:, 1]
final_test_probs = calibrated_meta.predict_proba(test_matrix)[:, 1]

# Threshold tuning
best_threshold = 0.5
best_global_f1 = 0

for thresh in np.arange(0.1, 0.9, 0.01):
    score = f1_score(y, (final_oof_probs > thresh).astype(int))
    if score > best_global_f1:
        best_global_f1 = score
        best_threshold = thresh

print("\nRESULTS :\n")
print(f"Base Stacking F1-Score (Thresh 0.50): {f1_score(y, (final_oof_probs > 0.5).astype(int)):.4f}")
print(f"Optimized F1-Score (Thresh {best_threshold:.2f}): {best_global_f1:.4f}")
print(f"Final OOF Accuracy: {accuracy_score(y, (final_oof_probs > best_threshold).astype(int)):.4f}")
print("\nClassification Report:\n", classification_report(y, (final_oof_probs > best_threshold).astype(int)))

# EXPORT
final_test_classes = (final_test_probs > best_threshold).astype(int)
submission = pd.DataFrame({
    'ID': test_ids,
    'CLASS': final_test_classes
})

print("Rows in submission:", len(submission))  # check row count
submission.to_csv('FINAL.csv', index=False)
print("\nFINAL.csv Created.")

Training stacking model...

RESULTS :

Base Stacking F1-Score (Thresh 0.50): 0.9877
Optimized F1-Score (Thresh 0.43): 0.9878
Final OOF Accuracy: 0.9903

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99     26465
           1       0.99      0.99      0.99     17311

    accuracy                           0.99     43776
   macro avg       0.99      0.99      0.99     43776
weighted avg       0.99      0.99      0.99     43776

Rows in submission: 10944

FINAL.csv Created.


In [6]:
import joblib

models = {
    "xgb_models": test_xgb,
    "cat_models": test_cat,
    "lgb_models": test_lgb,
    "meta_model": calibrated_meta,
    "best_threshold": best_threshold
}

joblib.dump(models, "full_pipeline.pkl")

['full_pipeline.pkl']